# MoE LLM Pipeline on Colab + Google Drive
This notebook mounts Drive, routes caches and artifacts to Drive, and runs the repo code from Colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/LLM_Pretraining_Pipeline'
RUN_ROOT = f'{DRIVE_ROOT}/colab_full_quality'
REPO_ROOT = '/content/LLM_Pretraining_Pipeline'

os.environ['LLM_PROJECT_ROOT'] = REPO_ROOT
os.environ['LLM_ARTIFACT_DIR'] = f'{RUN_ROOT}/artifacts'
os.environ['LLM_LOG_DIR'] = f'{RUN_ROOT}/artifacts/logs'
os.environ['LLM_RAW_DIR'] = f'{RUN_ROOT}/artifacts/data/raw'
os.environ['LLM_PROCESSED_DIR'] = f'{RUN_ROOT}/artifacts/data/processed'
os.environ['LLM_TOKENIZER_PATH'] = f'{RUN_ROOT}/artifacts/tokenizer/tokenizer.json'
os.environ['HF_HOME'] = f'{RUN_ROOT}/hf_cache'
os.environ['HF_DATASETS_CACHE'] = f'{RUN_ROOT}/hf_cache/datasets'
os.environ['TRANSFORMERS_CACHE'] = f'{RUN_ROOT}/hf_cache/transformers'
os.environ['LLM_STREAMING_CACHE_DIR'] = os.environ['HF_DATASETS_CACHE']
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

for path in [
    DRIVE_ROOT,
    RUN_ROOT,
    os.environ['LLM_ARTIFACT_DIR'],
    os.environ['LLM_LOG_DIR'],
    os.environ['LLM_RAW_DIR'],
    os.environ['LLM_PROCESSED_DIR'],
    os.environ['HF_HOME'],
    os.environ['HF_DATASETS_CACHE'],
    os.environ['TRANSFORMERS_CACHE'],
]:
    os.makedirs(path, exist_ok=True)

print('Artifacts ->', os.environ['LLM_ARTIFACT_DIR'])
print('Logs ->', os.environ['LLM_LOG_DIR'])
print('HF cache ->', os.environ['HF_DATASETS_CACHE'])

In [ ]:
import os

REPO_URL = 'https://github.com/your-user/LLM_Pretraining_Pipeline.git'
if not os.path.exists(REPO_ROOT):
    !git clone {REPO_URL} {REPO_ROOT}
%cd {REPO_ROOT}
!pip install -r requirements.txt

In [ ]:
import json
import os
import random
from pathlib import Path

from datasets import load_dataset

CONFIG = 'configs/profiles/colab_full_quality_drive_spec.yaml'
RAW_DIR = Path(os.environ['LLM_RAW_DIR'])
RAW_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
FINEWEB_SHARDS = 4
MAX_PRETRAIN_ROWS = None
MAX_SFT_ROWS = None
MAX_DPO_ROWS = None
MAX_GRPO_ROWS = None
EST_FINEWEB_DOWNLOAD_GB = FINEWEB_SHARDS * 2.15
random.seed(RANDOM_SEED)

def write_jsonl(path, rows):
    with Path(path).open('w', encoding='utf-8') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=True) + '\n')

def append_jsonl_row(path, row):
    with Path(path).open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(row, ensure_ascii=True) + '\n')

def to_text(value):
    if value is None:
        return ''
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        return '\n'.join(to_text(item) for item in value if to_text(item)).strip()
    if isinstance(value, dict):
        for key in ('content', 'text', 'response', 'answer', 'value'):
            if key in value:
                return to_text(value[key])
        return ' '.join(to_text(item) for item in value.values() if to_text(item)).strip()
    return str(value).strip()

def add_pretrain(text, source):
    global PRETRAIN_COUNT
    text = to_text(text)
    if len(text) >= 80 and (MAX_PRETRAIN_ROWS is None or PRETRAIN_COUNT < MAX_PRETRAIN_ROWS):
        append_jsonl_row(RAW_DIR / 'pretrain.jsonl', {'text': text, 'source': source})
        PRETRAIN_COUNT += 1

def add_sft(prompt, response, source):
    prompt = to_text(prompt)
    response = to_text(response)
    if prompt and response and (MAX_SFT_ROWS is None or len(sft_rows) < MAX_SFT_ROWS):
        sft_rows.append({'prompt': prompt, 'response': response, 'source': source})

def add_dpo(prompt, chosen, rejected, source):
    prompt = to_text(prompt)
    chosen = to_text(chosen)
    rejected = to_text(rejected)
    if prompt and chosen and rejected and chosen != rejected and (MAX_DPO_ROWS is None or len(dpo_rows) < MAX_DPO_ROWS):
        dpo_rows.append({'prompt': prompt, 'chosen': chosen, 'rejected': rejected, 'source': source})

def add_grpo(prompt, reference, source):
    prompt = to_text(prompt)
    reference = to_text(reference)
    if prompt and reference and (MAX_GRPO_ROWS is None or len(grpo_rows) < MAX_GRPO_ROWS):
        grpo_rows.append({'prompt': prompt, 'reference': reference, 'source': source})

PRETRAIN_COUNT = 0
for filename in ('pretrain.jsonl', 'sft.jsonl', 'dpo.jsonl', 'grpo.jsonl'):
    path = RAW_DIR / filename
    if path.exists():
        path.unlink()
pretrain_rows = []
sft_rows = []
dpo_rows = []
grpo_rows = []
fineweb_files = [f'sample/10BT/{idx:03d}_00000.parquet' for idx in range(FINEWEB_SHARDS)]
try:
    fineweb = load_dataset('HuggingFaceFW/fineweb-edu', 'sample-10BT', data_files=fineweb_files, split='train')
    for row in fineweb:
        add_pretrain(row.get('text'), f'fineweb-edu:{FINEWEB_SHARDS}_shard')
        if MAX_PRETRAIN_ROWS is not None and PRETRAIN_COUNT >= MAX_PRETRAIN_ROWS:
            break
except Exception as exc:
    print('FineWeb limited-shard load failed; continuing with fallback corpora:', repr(exc))

try:
    wikitext = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
    for row in wikitext:
        add_pretrain(row.get('text'), 'wikitext-2-raw-v1')
        if MAX_PRETRAIN_ROWS is not None and PRETRAIN_COUNT >= MAX_PRETRAIN_ROWS:
            break
except Exception as exc:
    print('Wikitext fallback failed:', repr(exc))

instruction_pairs = []
try:
    dolly = load_dataset('databricks/databricks-dolly-15k', split='train')
    for row in dolly:
        prompt = to_text(row.get('instruction'))
        context = to_text(row.get('context'))
        response = to_text(row.get('response'))
        if context:
            prompt = f'{prompt}\n\nContext: {context}'
        if prompt and response:
            instruction_pairs.append((prompt, response, 'databricks-dolly-15k'))
            add_pretrain(response, 'databricks-dolly-15k_response')
except Exception as exc:
    print('Dolly load failed:', repr(exc))

try:
    alpaca = load_dataset('tatsu-lab/alpaca', split='train')
    for row in alpaca:
        prompt = to_text(row.get('instruction'))
        input_text = to_text(row.get('input'))
        response = to_text(row.get('output'))
        if input_text:
            prompt = f'{prompt}\n\nInput: {input_text}'
        if prompt and response:
            instruction_pairs.append((prompt, response, 'tatsu-lab/alpaca'))
            add_pretrain(response, 'alpaca_response')
except Exception as exc:
    print('Alpaca load failed:', repr(exc))

random.shuffle(instruction_pairs)
for prompt, response, source in instruction_pairs:
    add_sft(prompt, response, source)
    add_grpo(prompt, response, source)
    if (MAX_SFT_ROWS is not None and len(sft_rows) >= MAX_SFT_ROWS) and (MAX_GRPO_ROWS is not None and len(grpo_rows) >= MAX_GRPO_ROWS):
        break

try:
    ultra = load_dataset('mlabonne/ultrafeedback-binarized-preferences-cleaned', split='train')
    for row in ultra:
        add_dpo(row.get('question'), row.get('chosen'), row.get('rejected'), 'ultrafeedback_cleaned')
        if MAX_DPO_ROWS is not None and len(dpo_rows) >= MAX_DPO_ROWS:
            break
except Exception as exc:
    print('UltraFeedback DPO load failed; using hard mismatched instruction negatives:', repr(exc))

if len(dpo_rows) < MAX_DPO_ROWS and instruction_pairs:
    for idx, (prompt, response, source) in enumerate(instruction_pairs):
        rejected = instruction_pairs[(idx + 997) % len(instruction_pairs)][1]
        add_dpo(prompt, response, rejected, f'{source}_hard_mismatch')
        if MAX_DPO_ROWS is not None and len(dpo_rows) >= MAX_DPO_ROWS:
            break

try:
    gsm8k = load_dataset('openai/gsm8k', 'main', split='train')
    for row in gsm8k:
        add_grpo(row.get('question'), row.get('answer'), 'gsm8k')
        if MAX_GRPO_ROWS is not None and len(grpo_rows) >= MAX_GRPO_ROWS:
            break
except Exception as exc:
    print('GSM8K GRPO load failed:', repr(exc))

random.shuffle(sft_rows)
random.shuffle(dpo_rows)
random.shuffle(grpo_rows)

write_jsonl(RAW_DIR / 'sft.jsonl', sft_rows)
write_jsonl(RAW_DIR / 'dpo.jsonl', dpo_rows)
write_jsonl(RAW_DIR / 'grpo.jsonl', grpo_rows)

EFFECTIVE_PRETRAIN_BATCH = 16
EFFECTIVE_SFT_BATCH = 8
EFFECTIVE_DPO_BATCH = 4
pretrain_train_rows = max(1, int(PRETRAIN_COUNT * 0.95))
sft_train_rows = max(1, int(len(sft_rows) * 0.95))
dpo_train_rows = max(1, int(len(dpo_rows) * 0.95))
grpo_train_rows = max(1, len(grpo_rows))
PRETRAIN_STEPS = max(1, (pretrain_train_rows + EFFECTIVE_PRETRAIN_BATCH - 1) // EFFECTIVE_PRETRAIN_BATCH)
SFT_STEPS = max(1, (sft_train_rows + EFFECTIVE_SFT_BATCH - 1) // EFFECTIVE_SFT_BATCH)
DPO_STEPS = max(1, (dpo_train_rows + EFFECTIVE_DPO_BATCH - 1) // EFFECTIVE_DPO_BATCH)
GRPO_STEPS = grpo_train_rows
PRETRAIN_WARMUP = min(500, max(10, PRETRAIN_STEPS // 20))
SFT_WARMUP = min(150, max(10, SFT_STEPS // 20))
DPO_WARMUP = min(80, max(10, DPO_STEPS // 20))
GRPO_WARMUP = min(50, max(10, GRPO_STEPS // 20))

config_text = f"""
runtime:
  profile_name: colab_full_quality_drive_spec
  project_root: .
  artifact_dir: artifacts
  log_dir: artifacts/logs
  seed: 42
  device: auto
  use_mixed_precision: true
  gradient_checkpointing: true

data:
  raw_dir: artifacts/data/raw
  processed_dir: artifacts/data/processed
  log_dir: artifacts/logs
  tokenizer_path: artifacts/tokenizer/tokenizer.json
  tokenizer_type: byte_level_bpe
  require_real_data: true
  use_streaming_sources: false
  pretrain_filename: pretrain.jsonl
  sft_filename: sft.jsonl
  dpo_filename: dpo.jsonl
  grpo_filename: grpo.jsonl
  max_vocab_size: 16000
  min_frequency: 2
  val_ratio: 0.05
  pretrain_seq_len: 384
  sft_seq_len: 384
  dpo_seq_len: 384
  min_chars: 50
  max_chars: 10000
  dedup: true
  strip_html: true
  lowercase: false
  quality_threshold: 0.38
  min_alpha_fraction: 0.58
  max_line_repeat_ratio: 0.22
  simhash_bits: 64
  simhash_bands: 4
  progress_every_rows: 1000
  checkpoint_every_rows: 5000
  flush_every_rows: 500

model:
  vocab_size: 16000
  max_seq_len: 384
  hidden_size: 512
  num_layers: 14
  mlp_hidden_size: 1408
  dropout: 0.0
  pad_token_id: 0
  bos_token_id: 1
  eos_token_id: 2
  tie_embeddings: true
  attn:
    num_heads: 8
    num_kv_heads: 4
    dropout: 0.0
    rope_base: 10000
  moe:
    enabled: true
    num_experts: 8
    top_k: 2
    capacity_factor: 1.25
    aux_loss_weight: 0.01
    shared_expert: true
    expert_hidden_mult: 2.5
    expert_interval: 2

train:
  batch_size: 1
  eval_batch_size: 1
  learning_rate: 0.0002
  weight_decay: 0.05
  adam_beta1: 0.9
  adam_beta2: 0.95
  adam_epsilon: 1.0e-08
  epochs: 1
  max_steps: 6000
  grad_accum_steps: 16
  grad_clip: 1.0
  log_every_steps: 10
  eval_every_steps: 500
  save_every_steps: 500
  warmup_steps: 500
  lr_scheduler: cosine
  min_lr_ratio: 0.1
  grpo_group_size: 3
  grpo_beta: 0.02
  grpo_max_new_tokens: 48
  resume_from:
"""
def write_stage_config(path, text, *, profile_name, max_steps, learning_rate, grad_accum_steps, warmup_steps, eval_every_steps, save_every_steps, grpo_group_size=3):
    stage_text = text
    stage_text = stage_text.replace('profile_name: colab_full_quality_drive_spec', f'profile_name: {profile_name}')
    stage_text = stage_text.replace('learning_rate: 0.0002', f'learning_rate: {learning_rate}')
    stage_text = stage_text.replace('max_steps: 6000', f'max_steps: {max_steps}')
    stage_text = stage_text.replace('grad_accum_steps: 16', f'grad_accum_steps: {grad_accum_steps}')
    stage_text = stage_text.replace('eval_every_steps: 500', f'eval_every_steps: {eval_every_steps}')
    stage_text = stage_text.replace('save_every_steps: 500', f'save_every_steps: {save_every_steps}')
    stage_text = stage_text.replace('warmup_steps: 500', f'warmup_steps: {warmup_steps}')
    stage_text = stage_text.replace('grpo_group_size: 3', f'grpo_group_size: {grpo_group_size}')
    Path(path).write_text(stage_text.strip() + '\n', encoding='utf-8')

PRETRAIN_CONFIG = CONFIG
SFT_CONFIG = 'configs/profiles/colab_full_quality_sft_spec.yaml'
DPO_CONFIG = 'configs/profiles/colab_full_quality_dpo_spec.yaml'
GRPO_CONFIG = 'configs/profiles/colab_full_quality_grpo_spec.yaml'

write_stage_config(PRETRAIN_CONFIG, config_text, profile_name='colab_full_quality_pretrain', max_steps=PRETRAIN_STEPS, learning_rate='0.0002', grad_accum_steps=16, warmup_steps=PRETRAIN_WARMUP, eval_every_steps=max(500, PRETRAIN_STEPS // 10), save_every_steps=max(500, PRETRAIN_STEPS // 10))
write_stage_config(SFT_CONFIG, config_text, profile_name='colab_full_quality_sft', max_steps=SFT_STEPS, learning_rate='0.0001', grad_accum_steps=8, warmup_steps=SFT_WARMUP, eval_every_steps=max(250, SFT_STEPS // 5), save_every_steps=max(250, SFT_STEPS // 5))
write_stage_config(DPO_CONFIG, config_text, profile_name='colab_full_quality_dpo', max_steps=DPO_STEPS, learning_rate='0.00003', grad_accum_steps=4, warmup_steps=DPO_WARMUP, eval_every_steps=max(200, DPO_STEPS // 5), save_every_steps=max(200, DPO_STEPS // 5))
write_stage_config(GRPO_CONFIG, config_text, profile_name='colab_full_quality_grpo', max_steps=GRPO_STEPS, learning_rate='0.00003', grad_accum_steps=1, warmup_steps=GRPO_WARMUP, eval_every_steps=max(100, GRPO_STEPS // 5), save_every_steps=max(100, GRPO_STEPS // 5), grpo_group_size=2)

print('Pretrain config ->', PRETRAIN_CONFIG)
print('SFT config ->', SFT_CONFIG)
print('DPO config ->', DPO_CONFIG)
print('GRPO config ->', GRPO_CONFIG)
print('FineWeb parquet shards requested ->', FINEWEB_SHARDS)
print('Estimated FineWeb parquet download GB ->', EST_FINEWEB_DOWNLOAD_GB)
print('Raw rows ->', {'pretrain': PRETRAIN_COUNT, 'sft': len(sft_rows), 'dpo': len(dpo_rows), 'grpo': len(grpo_rows)})
print('Full-data train steps ->', {'pretrain': PRETRAIN_STEPS, 'sft': SFT_STEPS, 'dpo': DPO_STEPS, 'grpo': GRPO_STEPS})

In [ ]:
!python -u -m src.data.prepare --config {PRETRAIN_CONFIG}
!python -u -m src.train.pretrain --config {PRETRAIN_CONFIG}
!python -u -m src.train.sft --config {SFT_CONFIG}
!python -u -m src.train.dpo --config {DPO_CONFIG}
!python -u -m src.train.grpo --config {GRPO_CONFIG}
!python -u -m src.eval.evaluate --config {GRPO_CONFIG} --stage grpo
!python -u -m src.inference.export --config {GRPO_CONFIG} --stage grpo
!tail -n 80 "$LLM_LOG_DIR/pipeline.log"